# 第2章 现金流、复利与贴现

> **核心问题**：今天的100元为什么通常不等于一年后的100元？如何在不同时间的现金流之间进行可解释的比较？

- 金融线：时间价值、现值、终值、净现值、通胀与定期投入。
- 数学线：指数、对数、极限与方程求根。
- Python线：函数参数、数组广播、表格、数值求根和交互滑块。

完成本章后，学生应能画现金流时间轴、区分增长与贴现、计算NPV，并说明利率不是脱离风险和期限的“常数”。

## AI学习状态

当前进度：第2章开始  
已掌握：简单收益率、函数和NumPy基础  
仍然薄弱：待填写  
下一步：每个公式先用一笔现金流手算。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 2.1 时间价值来自什么？

今天的资金可以用于消费、偿债或投资；未来现金流还包含通胀、违约、流动性和不确定性。因此跨期比较需要一个**与期限、风险和计价规则相匹配的利率**。

固定年利率 $r$、每年复利一次时：

$$FV=PV(1+r)^n,\qquad PV=\frac{FV}{(1+r)^n}$$

增长把今天推向未来；贴现把未来拉回今天。两者是同一关系的相反方向。

### 运行前预测

一年后的105元，在5%贴现率下今天值多少？如果贴现率提高，现值应上升还是下降？

### 我的预测

<!-- 在这里填写；完成前AI不要代答 -->

In [ ]:
present_value = 100
rate = 0.05
years = 3

future_value = present_value * (1 + rate) ** years
recovered_pv = future_value / (1 + rate) ** years
print({"终值": round(future_value, 2), "贴现回来的现值": round(recovered_pv, 2)})

**Python提示：用关键字参数减少歧义**

金融函数常有多个同为数值的参数。调用时写成`pv(amount=..., rate=..., years=...)`，比只写位置更不容易把期限和利率放反。

In [ ]:
def fv(amount, rate, years, compounds_per_year=1):
    # 名义年利率rate，每年复利compounds_per_year次。
    periods = years * compounds_per_year
    return amount * (1 + rate / compounds_per_year) ** periods


def pv(amount, rate, years, compounds_per_year=1):
    return amount / (1 + rate / compounds_per_year) ** (years * compounds_per_year)


print(fv(amount=1_000, rate=0.06, years=2))
print(pv(amount=1_123.60, rate=0.06, years=2))

## 2.2 复利频率与连续复利

名义年利率为 $r$、每年复利 $m$ 次：

$$FV=PV\left(1+\frac{r}{m}\right)^{mn}$$

当 $m\to\infty$ 时得到连续复利：

$$FV=PV e^{rn}$$

这正是高等数学中极限与指数函数的金融应用。

In [ ]:
frequencies = np.array([1, 2, 4, 12, 365, 10_000])
values = np.array([fv(1_000, 0.08, 1, int(m)) for m in frequencies])
continuous = 1_000 * np.exp(0.08)

comparison = pd.DataFrame({"每年复利次数": frequencies, "一年后终值": values})
comparison.loc[len(comparison)] = [np.inf, continuous]
comparison

### 观察问题

1. 复利频率增加时终值如何变化？
2. 为什么频率从365提高到10000的影响已经很小？
3. “8%连续复利”和“8%按年复利”是否是相同的实际收益？

### 我的回答

<!-- 在这里填写；完成前AI不要代答 -->

## 2.3 多笔现金流与净现值

对时间 $t=0,1,\ldots,T$ 的现金流 $CF_t$，净现值为：

$$NPV=\sum_{t=0}^{T}\frac{CF_t}{(1+r)^t}$$

$CF_0$通常是今天的投入，站在投资者角度常记为负数。NPV不是利润预测，它是**在给定贴现率和现金流假设下**的价值比较。

In [ ]:
cash_flows = np.array([-10_000, 3_000, 4_000, 5_000], dtype=float)
times = np.arange(len(cash_flows))
discount_rate = 0.08
discounted = cash_flows / (1 + discount_rate) ** times

table = pd.DataFrame({
    "年份": times,
    "现金流": cash_flows,
    "贴现因子": 1 / (1 + discount_rate) ** times,
    "现金流现值": discounted,
})
display(table.style.format({"现金流": "{:,.2f}", "贴现因子": "{:.4f}", "现金流现值": "{:,.2f}"}))
print("NPV：", round(discounted.sum(), 2))

**量化编程警告：单位和间隔必须一致**

年现金流应配年利率，月现金流应配月利率。不能把“8%年利率”直接用于每个月的贴现，也不能在未说明规则时把年利率简单除以12。

In [ ]:
rates = np.linspace(0, 0.25, 101)
npvs = np.array([np.sum(cash_flows / (1 + r) ** times) for r in rates])

fig, ax = plt.subplots()
ax.plot(rates, npvs)
ax.axhline(0, color="black", linewidth=1)
ax.set(title="贴现率变化如何影响净现值", xlabel="贴现率", ylabel="NPV（元）")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
plt.show()

### 观察问题

为什么这组现金流的NPV随贴现率上升而下降？如果未来现金流中包含负数，这一关系是否一定保持？

### 我的回答

<!-- 在这里填写；完成前AI不要代答 -->

## 2.4 内部收益率是“使NPV为0的利率”

$$0=\sum_{t=0}^{T}\frac{CF_t}{(1+IRR)^t}$$

IRR是方程的根，不是保证收益。现金流多次改变符号时可能有多个根或没有合适的根，因此必须结合NPV和经济含义使用。

In [ ]:
from scipy.optimize import brentq

def npv(rate, cash_flows):
    values = np.asarray(cash_flows, dtype=float)
    t = np.arange(values.size)
    return np.sum(values / (1 + rate) ** t)


irr = brentq(lambda r: npv(r, cash_flows), -0.99, 2.0)
print(f"IRR：{irr:.2%}")
print(f"代回后的NPV：{npv(irr, cash_flows):.8f}")

**Python提示：把函数传给函数**

`brentq`接收一个函数并寻找其零点。`lambda r: ...`临时定义“输入利率、输出NPV”的函数。数值算法给出近似根，所以代回结果接近0而不一定数学上完全等于0。

## 2.5 通胀与实际购买力

名义金额增长不代表购买力同幅增长。若名义收益率为 $r_n$、通胀率为 $\pi$，精确实际收益率为：

$$1+r_{real}=\frac{1+r_n}{1+\pi}$$

小比例时常近似为 $r_{real}\approx r_n-\pi$，但近似不是恒等式。

In [ ]:
nominal_rate = 0.05
inflation = 0.03
exact_real = (1 + nominal_rate) / (1 + inflation) - 1
approximation = nominal_rate - inflation

print({"精确实际收益率": f"{exact_real:.4%}", "近似值": f"{approximation:.4%}"})

t = np.arange(0, 31)
nominal_wealth = 10_000 * (1 + nominal_rate) ** t
purchasing_power = nominal_wealth / (1 + inflation) ** t

plt.plot(t, nominal_wealth, label="名义财富")
plt.plot(t, purchasing_power, label="按今天价格衡量的购买力")
plt.xlabel("年份"); plt.ylabel("元"); plt.title("名义增长与实际购买力"); plt.legend(); plt.show()

## 2.6 交互实验：贴现率、期限和现值

拖动参数，观察贴现率与期限如何共同影响10000元未来现金流的现值。

In [ ]:
def plot_discount(rate=0.05, years=10):
    t = np.arange(years + 1)
    values = 10_000 / (1 + rate) ** t
    plt.plot(t, values, marker="o")
    plt.title(f"未来10000元在不同期限的现值（贴现率={rate:.1%}）")
    plt.xlabel("距离今天的年数"); plt.ylabel("现值（元）"); plt.show()
    print({"rate": rate, "years": years, "pv_at_horizon": round(float(values[-1]), 2)})

try:
    from ipywidgets import interact, FloatSlider, IntSlider
    interact(plot_discount,
             rate=FloatSlider(value=0.05, min=0, max=0.20, step=0.01, description="贴现率"),
             years=IntSlider(value=10, min=1, max=30, description="期限"))
except ImportError:
    plot_discount()

## 2.7 编程练习：实现通用NPV

要求：接受现金流列表和贴现率；返回浮点数；拒绝`rate <= -1`。不要调用上面已经写好的`npv`。

In [ ]:
def student_npv(rate, cash_flows):
    # TODO：完成输入检查、时间数组和贴现求和
    return None

In [ ]:
answer = student_npv(0.10, [-100, 60, 60])
if answer is None:
    print("练习尚未完成。")
else:
    expected = -100 + 60 / 1.1 + 60 / 1.1**2
    print("基础测试通过：", np.isclose(answer, expected))

### 我的模型说明

为什么本函数默认现金流发生在等间隔期末？如果日期不规则，时间变量应该怎样改变？

<!-- 在这里填写；完成前AI不要代答 -->

### AI批改区

<!-- 检查单位、符号、边界条件和输入类型，不要覆盖学生答案。 -->

## 本章总结与小项目

建立一个“大学四年现金流比较器”：至少包含学费支出、兼职收入和毕业后收入两个情景；明确贴现率是假设；绘制现金流与累计现值；比较NPV但不把它解释为人生决策的唯一标准。

**关键结论**：增长与贴现互为逆过程；NPV依赖现金流和贴现率假设；名义财富必须与购买力区分；数值解必须代回检查。

**参考**：Investor.gov Compound Interest Calculator；Python/NumPy/SciPy官方文档。